In [3]:
!git clone --recursive https://github.com/naver/mast3r.git
%cd mast3r

!pip install -r requirements.txt
!pip install gradio
!pip install trimesh
!pip install roma

!pip install open3d shapely trimesh scipy roma pytorch3d

%cd /content


fatal: destination path 'mast3r' already exists and is not an empty directory.
/content/mast3r
  Using cached open3d-0.19.0-cp312-cp312-manylinux_2_31_x86_64.whl.metadata (4.3 kB)
ERROR: Could not find a version that satisfies the requirement pytorch3d (from versions: none)
ERROR: No matching distribution found for pytorch3d
/content


In [4]:
import os
import sys
import torch
import numpy as np
import glob

sys.path.append('/content/dust3r')
sys.path.append('/content/mast3r/dust3r')

from mast3r.model import AsymmetricMASt3R
from mast3r.image_pairs import make_pairs
from dust3r.inference import inference
from dust3r.cloud_opt import global_aligner
from dust3r.utils.image import load_images

ModuleNotFoundError: No module named 'mast3r.model'

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = "naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric"
model = AsymmetricMASt3R.from_pretrained(model_name).to(device)

In [ ]:
!gdown --folder 1ivpuKwZ25BosAlM9-YnROKlNPpPk-6Op

# 방별 이미지 넣기

In [ ]:
import os
os.listdir("/content/02_Data")

In [ ]:

import glob

target_folder = "/content/02_Data/"
target_image = "test1"

target = target_folder+target_image
img_paths = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
    img_paths.extend(glob.glob(os.path.join(target, ext)))

img_paths = sorted(img_paths)
for path in img_paths:
    print(f"  - {path}")

images = load_images(img_paths, size=512)



pointmaps / pts3d - 3D 공간 좌표
pixel_4d_matrix - 4차원 공간 데이터 매트릭스

In [ ]:

# 3. 이미지 쌍 빌드 및 기하학 추론
pairs = make_pairs(images, scene_graph='complete', prefilter=None, symmetrize=True)
with torch.no_grad():
    output = inference(pairs, model, device, batch_size=2)

# 4. 글로벌 정렬 최적화
scene = global_aligner(output, device=device)
scene.compute_global_alignment(niter=300, init="mst")

pointmaps = scene.get_pts3d()
confidences = scene.get_conf()

for i in range(len(img_paths)):
    pts3d = pointmaps[i].detach().cpu().numpy()     # (512, 512, 3)
    conf = confidences[i].detach().cpu().numpy()       # (512, 512)
    conf_expanded = np.expand_dims(conf, axis=-1)    # (512, 512, 1)

    # 4차원 결합
    pixel_4d_matrix = np.concatenate([pts3d, conf_expanded], axis=-1)   # (512, 512, 4)

# glb저장

In [ ]:
import os
import shutil
import struct
import json
import torch
import numpy as np
from google.colab import files

def get_3D_model_from_scene(outdir, scene, min_conf_thr=3, as_pointcloud=True, silent=False):
    """ GPU 텐서 및 동적 속성을 완벽히 처리하여 OOM과 에러를 방지하는 최종 빌더 """

    # 1. 3D 포인트 및 신뢰도 안전하게 CPU로 추출
    pts3d = [p.detach().cpu().numpy() if torch.is_tensor(p) else p for p in scene.get_pts3d()]
    conf = [f.detach().cpu().numpy() if torch.is_tensor(f) else f for f in scene.get_conf()]

    # 2. 🔥 [진짜 해결책] 색상 속성 동적 추출 및 GPU -> CPU 안전 이송
    colors = None
    color_candidates = ['colors', 'rgb', 'get_colors', 'get_rgb', 'im_colors', 'im_rgb']

    for cand in color_candidates:
        if hasattr(scene, cand):
            attr = getattr(scene, cand)
            raw_colors = attr() if callable(attr) else attr

            # 리스트나 텐서 내부의 요소들을 하나씩 CPU 넘파이로 안전하게 세탁
            if isinstance(raw_colors, (list, tuple)):
                colors = [c.detach().cpu().numpy() if torch.is_tensor(c) else c for c in raw_colors]
            elif torch.is_tensor(raw_colors):
                colors = [raw_colors.detach().cpu().numpy()]
            else:
                colors = raw_colors

            if not silent:
                print(f"🔍 진짜 색상 매핑 성공: '{cand}' 속성에서 데이터 복구 완료!")
            break

    if colors is None:
        print("⚠️ 색상 속성을 최종 감지하지 못해 순수 흰색 더미로 대체합니다.")
        colors = [np.ones_like(p) for p in pts3d]

    valid_pts = []
    valid_cols = []

    # 3. 신뢰도 기준(3.0) 필터링 진행
    for p, c, f in zip(pts3d, colors, conf):
        mask = (f >= min_conf_thr)
        valid_pts.append(p[mask])
        valid_cols.append(c[mask])

    final_pts = np.concatenate(valid_pts, axis=0).astype(np.float32)
    raw_cols = np.concatenate(valid_cols, axis=0)

    # 색상 포맷 uint8 0~255로 최종 압축
    if raw_cols.dtype == np.uint8:
        final_cols = raw_cols
    else:
        # 만약 0~1 사이의 float 형태라면 255를 곱해줍니다.
        if raw_cols.max() <= 1.01:
            final_cols = (raw_cols * 255.0).clip(0, 255).astype(np.uint8)
        else:
            final_cols = raw_cols.clip(0, 255).astype(np.uint8)

    num_vertices = final_pts.shape[0]

    if not silent:
        print(f"📊 필터링된 총 3D 포인트 개수: {num_vertices}개")

    # 4. 순수 3D GLB 바이너리 인코딩
    pts_bytes = final_pts.tobytes()
    cols_bytes = final_cols.tobytes()
    total_buffer_bytes = len(pts_bytes) + len(cols_bytes)

    gltf_json = {
        "asset": {"version": "2.0"},
        "scene": 0,
        "scenes": [{"nodes": [0]}],
        "nodes": [{"mesh": 0}],
        "meshes": [{"primitives": [{"attributes": {"POSITION": 0, "COLOR_0": 1}, "mode": 0}]}],
        "bufferViews": [
            {"buffer": 0, "byteOffset": 0, "byteLength": len(pts_bytes), "target": 34962},
            {"buffer": 0, "byteOffset": len(pts_bytes), "byteLength": len(cols_bytes), "target": 34962}
        ],
        "accessors": [
            {"bufferView": 0, "byteOffset": 0, "componentType": 5126, "count": num_vertices, "type": "VEC3", "max": final_pts.max(axis=0).tolist(), "min": final_pts.min(axis=0).tolist()},
            {"bufferView": 1, "byteOffset": 0, "componentType": 5121, "count": num_vertices, "type": "VEC3", "normalized": True}
        ],
        "buffers": [{"byteLength": total_buffer_bytes}]
    }

    json_str = json.dumps(gltf_json, separators=(',', ':')).encode('utf-8')
    json_pad = (4 - (len(json_str) % 4)) % 4
    json_str += b' ' * json_pad

    header = struct.pack('<4sII', b'glTF', 2, 12 + 8 + len(json_str) + 8 + total_buffer_bytes)
    chunk_json = struct.pack('<II', len(json_str), 0x4E4F534A) + json_str
    chunk_bin = struct.pack('<II', total_buffer_bytes, 0x004E4942) + pts_bytes + cols_bytes

    output_path = os.path.join(outdir, "scene.glb")
    if os.path.exists(output_path):
        os.remove(output_path)

    with open(output_path, 'wb') as f:
        f.write(header + chunk_json + chunk_bin)

    return output_path


# 💡 [안전장치] 파일명 변수 바인딩
if 'target_image' not in locals() and 'target_image' not in globals():
    target_image = "mast3r_room_scene"


# =================================================================
# 🎯 재생성 파이프라인 가동
# =================================================================

print("\n================== 📊 EXTRACTED DATA ==================")
poses = scene.get_im_poses().detach().cpu().numpy()
for i, pose in enumerate(poses):
    print(f"📷 이미지 [{i}]를 찍은 카메라 3D 위치 (4x4 Matrix):\n{pose}\n")
print("=======================================================\n")

print("💾 3D 공간 데이터를 고화질 GLB 파일로 굽는 중...")
output_filepath = get_3D_model_from_scene(
    outdir=".",
    silent=False,
    scene=scene,
    min_conf_thr=3,
    as_pointcloud=True
)

# 파일 이름을 target_image에 맞게 고정하고 다운로드 연동
if os.path.exists("./scene.glb"):
    final_path = f"./{target_image}.glb"
    if os.path.exists(final_path):
        os.remove(final_path)
    shutil.move("./scene.glb", final_path)
    print(f"🎉 3D 복원 완료! 파일 다운로드를 시작합니다: {final_path}")
    files.download(final_path)
else:
    print("./scene.glb 파일을 찾지 못했습니다")


================== 📊 EXTRACTED DATA ==================
📷 이미지 [0]를 찍은 카메라 3D 위치 (4x4 Matrix):
[[ 0.99994886  0.00955522 -0.00331372  0.        ]
 [-0.00957342  0.9999389  -0.00552072  0.        ]
 [ 0.00326077  0.00555216  0.99997926  0.        ]
 [ 0.          0.          0.          1.        ]]

📷 이미지 [1]를 찍은 카메라 3D 위치 (4x4 Matrix):
[[ 0.63409734  0.00747732  0.7732173  -0.83690643]
 [-0.1277954   0.9872159   0.09525523 -0.05282867]
 [-0.76262    -0.15921468  0.62694645  0.3723034 ]
 [ 0.          0.          0.          1.        ]]

📷 이미지 [2]를 찍은 카메라 3D 위치 (4x4 Matrix):
[[ 0.07675624  0.08771856  0.9931838  -0.63678956]
 [-0.12644409  0.9889361  -0.07757144  0.02263083]
 [-0.9889998  -0.11962812  0.08699846  0.54466575]
 [ 0.          0.          0.          1.        ]]


💾 3D 공간 데이터를 고화질 GLB 파일로 굽는 중...
⚠️ 색상 속성을 최종 감지하지 못해 순수 흰색 더미로 대체합니다.
📊 필터링된 총 3D 포인트 개수: 366개
🎉 3D 복원 완료! 파일 다운로드를 시작합니다: ./test1.glb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>